In [22]:
%pip install --upgrade gradio


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 338.5 kB/s eta 0:00:0000:0100:03
  Attempting uninstall: gradio
    Found existing installation: gradio 6.10.0
    Uninstalling gradio-6.10.0:
      Successfully uninstalled gradio-6.10.0

[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install python-dotenv requests gradio pandas

Endpoint URL: https://dbc-47f44256-85a9.cloud.databricks.com/serving-endpoints/agents_workspace-default-healthcare_agent/invocations
Token set : Yes


In [23]:
from dotenv import load_dotenv
import os

load_dotenv()   

DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
AGENT_ENDPOINT = os.getenv("AGENT_ENDPOINT")

HEADERS = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

print(f"Endpoint URL: {AGENT_ENDPOINT}")
print(f"Token set : {'Yes' if DATABRICKS_TOKEN else 'NO - check .env file'}")

Endpoint URL: https://dbc-47f44256-85a9.cloud.databricks.com/serving-endpoints/agents_workspace-default-healthcare_agent/invocations
Token set : Yes


# Core function

In [24]:
import requests
import json

def chat_with_agent(question: str) -> dict:
    # Construct payload using the agent's expected schema
    payload = {
        "dataframe_records": [
            {
                "input": [{"role": "user", "content": question}],
                "custom_inputs": {"session_id": "ui-testing-session"}
            }
        ]
    }
    
    response = requests.post(
        AGENT_ENDPOINT,
        headers = HEADERS,
        json    = payload,
        timeout = 180
    )

    if response.status_code != 200:
        # Try fallback payload format widely used by MLflow 2.x
        if "BAD_REQUEST" in response.text or "MALFORMED_REQUEST" in response.text or "schema" in response.text.lower():
            payload_fallback = {
                "inputs": {
                    "input": [{"role": "user", "content": question}],
                    "custom_inputs": {"session_id": "ui-testing-session"}
                }
            }
            response = requests.post(
                AGENT_ENDPOINT,
                headers = HEADERS,
                json    = payload_fallback,
                timeout = 180
            )
            if response.status_code != 200:
                return {"error": f"HTTP {response.status_code}: {response.text}"}
        else:
            return {"error": f"HTTP {response.status_code}: {response.text}"}

    data = response.json()
    answer = ""
    try:
        # Parse standard Databricks Agent response (OpenAI format or MLflow Pyfunc format)
        if "choices" in data and isinstance(data["choices"], list) and len(data["choices"]) > 0:
            answer = data["choices"][0].get("message", {}).get("content", "")
        elif "predictions" in data:
            preds = data["predictions"]
            extracted = False
            
            # Handle the specific Databricks complex nested response containing 'reasoning' and 'message'
            if isinstance(preds, dict) and "output" in preds and isinstance(preds["output"], list):
                texts = []
                for item in preds["output"]:
                    if isinstance(item, dict) and item.get("type") == "message" and "content" in item:
                        content = item["content"]
                        if isinstance(content, str):
                            texts.append(content)
                        elif isinstance(content, list):
                            for sub in content:
                                if isinstance(sub, dict) and "text" in sub:
                                    texts.append(sub["text"])
                                elif isinstance(sub, str):
                                    texts.append(sub)
                if texts:
                    answer = "\n".join(texts)
                    extracted = True
            
            if not extracted:
                if isinstance(preds, list) and len(preds) > 0:
                    pred = preds[0]
                    if isinstance(pred, dict):
                         answer = pred.get("response", pred.get("content", pred.get("answer", pred.get("output", str(pred)))))
                    else:
                         answer = str(pred)
                elif isinstance(preds, dict):
                    # If predictions is a dict (e.g., column based output like {"output": ["..."]})
                    if "output" in preds and isinstance(preds["output"], list) and len(preds["output"]) > 0:
                         answer = str(preds["output"][0])
                    elif "response" in preds and isinstance(preds["response"], list) and len(preds["response"]) > 0:
                         answer = str(preds["response"][0])
                    else:
                         answer = str(preds)
                else:
                    answer = str(preds)
        else:
            answer = str(data)
    except Exception as e:
        answer = f"Response Parsing Error: {str(e)}\n\nRaw Data: {json.dumps(data, indent=2)}"

    return {
        "answer": answer,
        "raw_data": data
    }

# Test to verify connection

In [ ]:
# ============================================================
# Quick test — run this first to verify connection
# ============================================================
result = chat_with_agent("How many hospitals have cardiology?")

if "error" in result:
    print(f"Error:\n{result['error']}")
else:
    print(f"Answer:\n{result['answer']}")
    print(f"\nRaw Response:\n{json.dumps(result['raw_data'], indent=2)}")


# Gradio UI

In [32]:
import gradio as gr
import json

print(f"Gradio version: {gr.__version__}")

def format_response(result: dict) -> str:
    """Build a rich markdown response that shows reasoning steps + final answer."""
    if "error" in result:
        return f"❌ **Error:** {result['error']}"

    raw = result.get("raw_data", {})
    preds = raw.get("predictions", {})
    output_items = []

    if isinstance(preds, dict) and "output" in preds:
        output_items = preds["output"]
    elif isinstance(preds, list):
        output_items = preds

    if not output_items:
        # Fallback: just return the extracted answer
        return result.get("answer", str(raw))

    sections = []
    step_num = 1

    for item in output_items:
        if not isinstance(item, dict):
            continue

        item_type = item.get("type", "")

        # ── Tool / Function Call ──
        if item_type == "function_call":
            tool_name = item.get("name", "unknown_tool")
            # Make tool name more readable
            display_name = tool_name.replace("workspace__default__", "").replace("_", " ").title()
            try:
                args = json.loads(item.get("arguments", "{}"))
                args_str = json.dumps(args, indent=2)
            except (json.JSONDecodeError, TypeError):
                args_str = item.get("arguments", "{}")

            sections.append(
                f"**Step {step_num}: 🔧 Tool Call → `{display_name}`**\n"
                f"```json\n{args_str}\n```"
            )
            step_num += 1

        # ── Tool / Function Call Output ──
        elif item_type == "function_call_output":
            try:
                output_data = json.loads(item.get("output", "{}"))
            except (json.JSONDecodeError, TypeError):
                output_data = item.get("output", "")

            if isinstance(output_data, dict):
                # Show SQL query if present
                parts = []
                if "sql_query" in output_data:
                    parts.append(f"**SQL Query:**\n```sql\n{output_data['sql_query']}\n```")
                if "results" in output_data:
                    parts.append(f"**Results:** `{json.dumps(output_data['results'])}`")
                if "answer" in output_data:
                    parts.append(f"**Tool Answer:** {output_data['answer']}")
                if "num_rows" in output_data:
                    parts.append(f"**Rows returned:** {output_data['num_rows']}")

                # If no known keys matched, show raw
                if not parts:
                    parts.append(f"```json\n{json.dumps(output_data, indent=2)}\n```")

                sections.append(
                    f"**Step {step_num}: 📊 Tool Result**\n" + "\n".join(parts)
                )
            else:
                sections.append(
                    f"**Step {step_num}: 📊 Tool Result**\n{output_data}"
                )
            step_num += 1

        # ── Reasoning ──
        elif item_type == "reasoning":
            summaries = item.get("summary", [])
            reasoning_texts = []
            for s in summaries:
                if isinstance(s, dict) and "text" in s:
                    reasoning_texts.append(s["text"])
                elif isinstance(s, str):
                    reasoning_texts.append(s)
            if reasoning_texts:
                reasoning_str = "\n".join(f"- {t}" for t in reasoning_texts)
                sections.append(
                    f"**Step {step_num}: 🧠 Reasoning**\n{reasoning_str}"
                )
                step_num += 1

        # ── Final Message ──
        elif item_type == "message":
            content = item.get("content", "")
            if isinstance(content, str):
                final_text = content
            elif isinstance(content, list):
                texts = []
                for sub in content:
                    if isinstance(sub, dict) and "text" in sub:
                        texts.append(sub["text"])
                    elif isinstance(sub, str):
                        texts.append(sub)
                final_text = "\n".join(texts)
            else:
                final_text = str(content)

            sections.append(f"---\n### ✅ Answer\n{final_text}")

    if sections:
        return "\n\n".join(sections)

    # Fallback
    return result.get("answer", str(raw))


def respond(message, history):
    result = chat_with_agent(message)
    return format_response(result)

chatbot = gr.Chatbot(
    height=600,
    render_markdown=True,
)

with gr.Blocks(title="Healthcare Agent Assistant") as demo:
    gr.Markdown("## 🏥 Healthcare Agent UI")
    gr.Markdown("Ask questions to the Healthcare Agent powered by Databricks.")

    gr.ChatInterface(
        fn       = respond,
        chatbot  = chatbot,
        examples = [
            ["What services does 2BN Military Hospital offer?"],
            ["Which hospitals have emergency care?"],
            ["How many hospitals have cardiology?"],
            ["Find facilities with maternity services"],
        ],
        title = "",
    )

demo.launch(share=False)


Gradio version: 6.10.0
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/usr/lib/python3/dist-packages/urllib3/connectionpool.py", line 446, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "/usr/lib/python3/dist-packages/urllib3/connectionpool.py", line 441, in _make_request
    httplib_response = conn.getresponse()
  File "/usr/lib/python3.10/http/client.py", line 1375, in getresponse
    response.begin()
  File "/usr/lib/python3.10/http/client.py", line 318, in begin
    version, status, reason = self._read_status()
  File "/usr/lib/python3.10/http/client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
  File "/usr/lib/python3.10/ssl.py", line 1303, in recv_into
    return self.read(nbytes, buffer)
  File "/usr/lib/python3.10/ssl.py", line 1159, in read
    return self._sslobj.read(len, buffer)
TimeoutError: The read operation ti